# Exploratory Data Analysis & Insights - Notebook Walkthrough

This notebook demonstrates how to utilize the `src` modular pipeline programmatically for data cleaning, statistical analysis, outlier detection, visual exploration, and report generation.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../'))

from src.data_loader import load_data, get_dataset_info
from src.cleaner import get_missing_summary, get_duplicate_summary, handle_missing_values, remove_duplicates
from src.analyzer import get_numerical_summary, get_categorical_summary, compute_correlation_matrix, get_top_correlations
from src.outliers import detect_all_outliers, handle_outliers
from src.insights import generate_automated_insights
from src.reporter import generate_markdown_report

## 1. Load Dataset

In [ ]:
dataset_path = '../data/sample_employee_data.csv'
df, msg = load_data(dataset_path)
print(f"Load Status: {msg}")
print(f"Shape: {df.shape}")
df.head()

## 2. Dataset Schema & Quality Audit

In [ ]:
info = get_dataset_info(df)
print(f"Numerical Columns: {info['numerical_cols']}")
print(f"Categorical Columns: {info['categorical_cols']}")

missing_df = get_missing_summary(df)
print("\n--- Missing Values Summary ---")
print(missing_df)

dup_info = get_duplicate_summary(df)
print(f"\nDuplicate Count: {dup_info['duplicate_count']} ({dup_info['duplicate_percent']}%)")

## 3. Data Cleaning Pipeline

In [ ]:
# Deduplicate data
df_clean = remove_duplicates(df)

# Impute missing values
df_clean = handle_missing_values(df_clean, strategy_map={
    'Performance_Score': 'median',
    'Education': 'mode'
})

print(f"Cleaned Shape: {df_clean.shape}")
print(f"Remaining Nulls: {df_clean.isnull().sum().sum()}")

## 4. Statistical Summary & Correlation Analysis

In [ ]:
num_stats = get_numerical_summary(df_clean)
display(num_stats)

corr_matrix = compute_correlation_matrix(df_clean)
top_corrs = get_top_correlations(corr_matrix, threshold=0.5)
display(top_corrs)

## 5. Outlier Detection & Treatment

In [ ]:
outliers = detect_all_outliers(df_clean, method='iqr', factor=1.5)
display(outliers)

# Cap outliers for Salary
df_clean = handle_outliers(df_clean, col='Salary', action='cap', method='iqr', factor=1.5)

## 6. Automated Insights & Report Generation

In [ ]:
insights = generate_automated_insights(df_clean)
print("=== AUTOMATED INSIGHTS ===")
for key, val in insights.items():
    print(f"\n[{key.upper()}]:")
    for line in val:
        print(f" - {line}")

report_md = generate_markdown_report(df_clean, dataset_name='Employee_Data_Cleaned', insights=insights)
with open('../reports/sample_eda_report.md', 'w') as f:
    f.write(report_md)
print("\nReport saved to ../reports/sample_eda_report.md")